# 切分 → 向量 → 写入 Elasticsearch

本笔记本是 [c03_embedding_smoke.ipynb](c03_embedding_smoke.ipynb) 的**下游**：在相同语料与滑窗参数下完成 **BGE-M3 编码**，将块 **bulk 写入** ES，并用 **kNN + 查询向量**做验收。

**前置（缺一不可）**

- **Python**：项目根 `.venv`；`uv sync`（默认含 **dev** → `elasticsearch` 客户端）；向量需 **`uv sync --extra embedding`**。
- **Elasticsearch**：本机 `9200` 可达；示例：`docker compose -f doc/storage/docker-compose.elasticsearch.yml up -d`。
- **`.env`**：`MODEL_*`、`BGE_M3_PATH`、`ES_HOST`/`ES_PORT` 等；与 c03 一样依赖配置层。

**索引隔离**：默认使用 `ES_INDEX` 加后缀 `_nb`（如 `rag_law_doc_chunks_nb`），避免覆盖你在其它流程里用的正式索引。可在下方将 `RECREATE_INDEX` 设为 `False` 以复用已有索引。


In [11]:
from __future__ import annotations

import os
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

os.chdir(_ROOT)
print("仓库根目录:", _ROOT)
print("已加入 sys.path:", _SRC)


仓库根目录: /Users/zheng/projects/python/ai/rag/rag_law
已加入 sys.path: /Users/zheng/projects/python/ai/rag/rag_law/src


## 0. 运行时检查（torch / FlagEmbedding / elasticsearch）


In [12]:
import importlib.util

print("当前 Python:", __import__("sys").executable)

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("未安装 torch。请执行: uv sync --extra embedding")

import torch

try:
    from FlagEmbedding import BGEM3FlagModel  # noqa: F401
except ModuleNotFoundError as e:
    raise RuntimeError("未安装 FlagEmbedding。请执行: uv sync --extra embedding") from e

try:
    import elasticsearch  # noqa: F401
except ModuleNotFoundError as e:
    raise RuntimeError("未安装 elasticsearch 客户端。请执行: uv sync（dev 已包含）") from e

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("elasticsearch 包 OK")


当前 Python: /Users/zheng/projects/python/ai/rag/rag_law/.venv/bin/python
torch: 2.11.0 | CUDA: False
elasticsearch 包 OK


## 1. 配置 `get_settings`


In [13]:
from conf.settings import get_settings, project_root

get_settings.cache_clear()
settings = get_settings()

print("es_url:", settings.es_url)
print("ES_INDEX (默认):", settings.es_index)
print("RETRIEVAL_K:", settings.retrieval_k)
print("BGE_M3_PATH:", settings.bge_m3_path)
print("EMBEDDING_BATCH_SIZE:", settings.embedding_batch_size)


es_url: http://localhost:9200
ES_INDEX (默认): rag_law_chunks
RETRIEVAL_K: 5
BGE_M3_PATH: /Users/zheng/jupyter/c10_edu_rag/rag_qa/models/bge-m3
EMBEDDING_BATCH_SIZE: 32


## 2. 笔记本专用索引名

- 默认：`NOTEBOOK_ES_INDEX = settings.es_index + "_nb"`。
- `RECREATE_INDEX=True` 会先删后建（**仅作用于上述笔记本索引**）。


In [14]:
# 与正式数据隔离；若要写入 settings.es_index 本身，改为 NOTEBOOK_ES_INDEX = settings.es_index（慎用）
NOTEBOOK_ES_INDEX = settings.es_index + "_nb"
RECREATE_INDEX = True
# 跑完最后一格验收后是否删除笔记本索引（False 保留便于 curl / Kibana 查看）
DELETE_INDEX_AT_END = False

print("将使用索引:", NOTEBOOK_ES_INDEX)
print("RECREATE_INDEX:", RECREATE_INDEX, "| DELETE_INDEX_AT_END:", DELETE_INDEX_AT_END)


将使用索引: rag_law_chunks_nb
RECREATE_INDEX: True | DELETE_INDEX_AT_END: False


## 3. 语料与切分（与 c03 一致）

参数：`CHUNK_SIZE=300`、`CHUNK_OVERLAP=40`、滑窗无句边界。可将 `SAMPLE_TEXT` 换为 `fixtures/宪法.md`（块多、编码慢）。


In [15]:
# --- 短样例（与 c03 相同）---
SAMPLE_TEXT = '''
中华人民共和国宪法（节选样式）

第一条 中华人民共和国是工人阶级领导的、以工农联盟为基础的人民民主专政的社会主义国家。
社会主义制度是中华人民共和国的根本制度。中国共产党领导是中国特色社会主义最本质的特征。

第二条 中华人民共和国的一切权力属于人民。

第五条 中华人民共和国实行依法治国，建设社会主义法治国家。
'''

# --- 可选：长语料 ---
# SAMPLE_TEXT = (project_root() / "tests" / "test_chunking" / "fixtures" / "宪法.md").read_text(encoding="utf-8")

full_text = SAMPLE_TEXT.strip()
print("len(full_text):", len(full_text))


len(full_text): 158


In [17]:
from chunking.split import TextChunk, iter_chunks_for_text

CHUNK_SIZE = 30
CHUNK_OVERLAP = 4

chunks: list[TextChunk] = list(
    iter_chunks_for_text(
        full_text,
        source_file="notebook_c04_sample.md",
        source_path="notebooks/c04_es.ipynb",
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        boundary_aware=False,
    )
)

print(f"块数: {len(chunks)}")
for c in chunks[:5]:
    print(f"  [#{c.chunk_index}] [{c.char_start}:{c.char_end}] len={len(c.text)}")

for c in chunks:
    assert full_text[c.char_start : c.char_end] == c.text
print("断言: 切片与 full_text 一致 OK")


块数: 6
  [#0] [0:30] len=30
  [#1] [26:56] len=30
  [#2] [52:82] len=30
  [#3] [78:108] len=30
  [#4] [104:134] len=30
断言: 切片与 full_text 一致 OK


## 4. 向量化（`embed_documents`）


In [18]:
from embeddings import build_embedder

texts_to_embed = [c.text for c in chunks]
assert len(texts_to_embed) == len(chunks)

embedder = build_embedder(settings)
embeddings: list[list[float]] = embedder.embed_documents(texts_to_embed)

dim = embedder.dense_dimension
print("dense_dimension:", dim)
assert len(embeddings) == len(chunks)
for i, row in enumerate(embeddings):
    assert len(row) == dim, f"行 {i} 维度异常"
print("断言: len(embeddings)==len(chunks)，且每行维度==dense_dimension OK")


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 81.59it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 17.92it/s]

dense_dimension: 1024
断言: len(embeddings)==len(chunks)，且每行维度==dense_dimension OK


## 5. 组装 ES 文档（与 `bulk_index_chunks` / `TextChunk` 对齐）

必填：`text`、`embedding`、`source_file`、`chunk_index`、`char_start`、`char_end`。其余由 `apply_chunk_source_defaults` 在入库时补齐（见 `es_store.chunk_defaults`）；此处显式写入 `source_doc_id`（`TextChunk.source_id`）、`mime_type`、`doc_type`、`domain`、`chunk_type`（默认 `text`）。


In [19]:
from es_store.chunk_defaults import apply_chunk_source_defaults

docs: list[dict] = []
for c, emb in zip(chunks, embeddings):
    row: dict = {
        "text": c.text,
        "embedding": emb,
        "source_file": c.source_file,
        "source_path": c.source_path or "",
        "chunk_index": c.chunk_index,
        "char_start": c.char_start,
        "char_end": c.char_end,
        "source_doc_id": c.source_id or "",
        "mime_type": c.mime_type,
        "doc_type": c.doc_type,
        "domain": c.domain,
        "chunk_type": "text",
    }
    if c.extra:
        row["extra"] = c.extra
    docs.append(apply_chunk_source_defaults(row))
print("待写入条数:", len(docs))
print("示例字段 keys:", sorted(docs[0].keys()) if docs else [])


待写入条数: 6
示例字段 keys: ['char_end', 'char_start', 'chunk_index', 'chunk_type', 'doc_type', 'domain', 'embedding', 'mime_type', 'source_doc_id', 'source_file', 'source_path', 'source_sha256', 'text']


## 6. 连接 ES、建索引、bulk 写入


In [20]:
from es_store.client import elasticsearch_client
from es_store.store import EsChunkStore

client = elasticsearch_client(settings)

if not client.ping():
    raise RuntimeError("无法连接 ES，请检查集群与 ES_HOST/ES_PORT")

store = EsChunkStore(client, NOTEBOOK_ES_INDEX, dense_dims=embedder.dense_dimension)
store.ensure_index(recreate=RECREATE_INDEX)

n_ok, errs = store.bulk_index_chunks(docs)
print("bulk 成功条数:", n_ok)
if errs:
    print("bulk 错误:", errs)
    raise RuntimeError("bulk 失败")

store.refresh()
cnt = client.count(index=NOTEBOOK_ES_INDEX)
total = int(cnt["count"])  # elasticsearch 返回 ObjectApiResponse，支持 __getitem__
print("索引文档数 count:", total)
assert total == len(docs), "count 应与块数一致"
print("断言: count == len(chunks) OK")


bulk 成功条数: 6
索引文档数 count: 6
断言: count == len(chunks) OK


## 7. 验收：查询向量 + kNN

`k` 取 `min(RETRIEVAL_K, 块数)`，避免块数少于 k 时行为费解。


In [21]:
query_text = "国家的根本制度是什么？"
query_vec = embedder.embed_query(query_text)
assert len(query_vec) == embedder.dense_dimension

k = min(settings.retrieval_k, len(chunks))
hits = store.search_knn(query_vec, k=k)
print(f"查询: {query_text!r} | k={k} | 命中数: {len(hits)}")
assert len(hits) <= k
for h in hits:
    src = h.get("source") or {}
    t = (src.get("text") or "")[:120].replace("\n", " ")
    print(f"  score={h['score']:.4f} id={h['id']!r} | {t}…")


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.20it/s]

查询: '国家的根本制度是什么？' | k=5 | 命中数: 5
  score=0.8513 id='notebook_c04_sample.md:2' | 的社会主义国家。 社会主义制度是中华人民共和国的根本制度。中…
  score=0.7780 id='notebook_c04_sample.md:3' | 制度。中国共产党领导是中国特色社会主义最本质的特征。  第二…
  score=0.7699 id='notebook_c04_sample.md:1' | 和国是工人阶级领导的、以工农联盟为基础的人民民主专政的社会主…
  score=0.7580 id='notebook_c04_sample.md:4' |   第二条 中华人民共和国的一切权力属于人民。  第五条 中…
  score=0.7517 id='notebook_c04_sample.md:5' | 五条 中华人民共和国实行依法治国，建设社会主义法治国家。…


## 8.（可选）删除笔记本索引

仅当 `DELETE_INDEX_AT_END=True` 时执行。


In [ ]:
if DELETE_INDEX_AT_END and client.indices.exists(index=NOTEBOOK_ES_INDEX):
    client.indices.delete(index=NOTEBOOK_ES_INDEX)
    print("已删除索引:", NOTEBOOK_ES_INDEX)
else:
    print("保留索引:", NOTEBOOK_ES_INDEX, "（DELETE_INDEX_AT_END=False 或索引不存在）")
